# Construction Analytics Dashboard
Interactive KPI summary for Project Cost, Schedule Performance, and Subcontractor Health.

In [ ]:
df_scorecard  = spark.read.format('delta').table('gold_portfolio_scorecards').limit(100000).toPandas()
df_cost       = spark.read.format('delta').table('gold_cost_analysis').limit(100000).toPandas()
df_sub        = spark.read.format('delta').table('gold_subcontractor_performance').limit(100000).toPandas()
df_alerts     = spark.read.format('delta').table('gold_overrun_alerts').limit(100000).toPandas()

In [ ]:
total_projects    = len(df_scorecard)
active_projects   = int((df_scorecard['status'] == 'In Progress').sum())
total_budget      = round(df_scorecard['budget'].sum(), 0)
total_actual_cost = round(df_scorecard['total_actual_cost'].sum(), 0) if 'total_actual_cost' in df_scorecard else 0
avg_schedule_var  = round(df_scorecard['schedule_variance_days'].mean(), 1)
critical_count    = int((df_scorecard['performance_band'] == 'Critical').sum())

print(f"Total Projects       : {total_projects}")
print(f"Active Projects      : {active_projects}")
print(f"Total Budget         : £{total_budget:,.0f}")
print(f"Avg Schedule Variance: {avg_schedule_var} days")
print(f"Critical Projects    : {critical_count}")

In [ ]:
# Top 10 projects by cost variance
top_overrun = (
    df_scorecard[['project_name', 'region', 'project_type', 'budget',
                  'schedule_variance_days', 'schedule_risk_band',
                  'cost_variance_pct', 'cost_risk_band', 'performance_band']]
    .sort_values('cost_variance_pct', ascending=False)
    .head(10)
)

# Subcontractor summary (top 10 by task volume)
top_subs = (
    df_sub[['company_name', 'trade', 'rating', 'total_tasks',
             'delayed_tasks', 'on_time_rate', 'avg_schedule_variance_days']]
    .sort_values('total_tasks', ascending=False)
    .head(10)
)

html = f"""
<!DOCTYPE html><html><head>
<meta charset="utf-8">
<title>Construction Analytics Dashboard</title>
<style>
  body {{ font-family: Segoe UI, Arial, sans-serif; background:#f4f6f9; margin:0; padding:20px; }}
  h1   {{ color:#0078d4; border-bottom:3px solid #0078d4; padding-bottom:8px; }}
  h2   {{ color:#323130; margin-top:30px; }}
  .kpis {{ display:flex; gap:16px; flex-wrap:wrap; margin:20px 0; }}
  .kpi  {{ background:#fff; border-radius:8px; padding:20px 28px; box-shadow:0 2px 6px rgba(0,0,0,.08);
           min-width:160px; text-align:center; }}
  .kpi .val {{ font-size:2em; font-weight:700; color:#0078d4; }}
  .kpi .lbl {{ font-size:.85em; color:#605e5c; margin-top:4px; }}
  table {{ border-collapse:collapse; width:100%; background:#fff; border-radius:8px;
           box-shadow:0 2px 6px rgba(0,0,0,.08); overflow:hidden; margin-bottom:30px; }}
  th    {{ background:#0078d4; color:#fff; padding:10px 14px; text-align:left; font-size:.9em; }}
  td    {{ padding:9px 14px; border-bottom:1px solid #edebe9; font-size:.88em; }}
  tr:hover td {{ background:#f3f9ff; }}
  .badge {{ display:inline-block; padding:2px 10px; border-radius:12px; font-size:.8em; font-weight:600; }}
  .Excellent       {{ background:#dff6dd; color:#107c10; }}
  .Good            {{ background:#cceaff; color:#0078d4; }}
  .Needs.Attention {{ background:#fff4ce; color:#b7610e; }}
  .Critical        {{ background:#fde7e9; color:#a80000; }}
  .On.Track        {{ background:#dff6dd; color:#107c10; }}
  .On.Budget       {{ background:#dff6dd; color:#107c10; }}
  .High            {{ background:#fff4ce; color:#b7610e; }}
  .Severe          {{ background:#fde7e9; color:#a80000; }}
</style></head><body>
<h1>🏗️ Construction Analytics Dashboard</h1>

<div class="kpis">
  <div class="kpi"><div class="val">{total_projects}</div><div class="lbl">Total Projects</div></div>
  <div class="kpi"><div class="val">{active_projects}</div><div class="lbl">Active Projects</div></div>
  <div class="kpi"><div class="val">£{total_budget:,.0f}</div><div class="lbl">Total Portfolio Budget</div></div>
  <div class="kpi"><div class="val">{avg_schedule_var}d</div><div class="lbl">Avg Schedule Variance</div></div>
  <div class="kpi"><div class="val">{critical_count}</div><div class="lbl">Critical Projects</div></div>
  <div class="kpi"><div class="val">{len(df_alerts)}</div><div class="lbl">Overrun Alerts</div></div>
</div>

<h2>Top 10 Projects by Cost Overrun</h2>
<table>
  <tr><th>Project</th><th>Region</th><th>Type</th><th>Budget</th><th>Schedule Var (d)</th><th>Schedule Risk</th><th>Cost Var %</th><th>Cost Risk</th><th>Performance</th></tr>
"""
for _, r in top_overrun.iterrows():
    pb = str(r['performance_band']).replace(' ', '.')
    sr = str(r['schedule_risk_band']).replace(' ', '.')
    cr = str(r['cost_risk_band']).replace(' ', '.')
    html += f"""
  <tr>
    <td>{r['project_name']}</td>
    <td>{r['region']}</td>
    <td>{r['project_type']}</td>
    <td>£{r['budget']:,.0f}</td>
    <td>{r['schedule_variance_days']}</td>
    <td><span class="badge {sr}">{r['schedule_risk_band']}</span></td>
    <td>{r['cost_variance_pct']:+.1f}%</td>
    <td><span class="badge {cr}">{r['cost_risk_band']}</span></td>
    <td><span class="badge {pb}">{r['performance_band']}</span></td>
  </tr>"""

html += """
</table>

<h2>Top 10 Subcontractors by Task Volume</h2>
<table>
  <tr><th>Company</th><th>Trade</th><th>Rating</th><th>Total Tasks</th><th>Delayed Tasks</th><th>On-Time Rate</th><th>Avg Delay (d)</th></tr>
"""
for _, r in top_subs.iterrows():
    html += f"""
  <tr>
    <td>{r['company_name']}</td>
    <td>{r['trade']}</td>
    <td>{'⭐' * int(round(r['rating']))}</td>
    <td>{int(r['total_tasks']):,}</td>
    <td>{int(r['delayed_tasks']):,}</td>
    <td>{r['on_time_rate']:.1f}%</td>
    <td>{r['avg_schedule_variance_days']:.1f}</td>
  </tr>"""

html += """
</table>
</body></html>"""

displayHTML(html)

In [ ]:
# Save dashboard HTML to lakehouse Files
mssparkutils.fs.put('Files/dashboard.html', html, overwrite=True)
print('Dashboard saved to Files/dashboard.html')